# 运维：在生产环境中运行 Managed Agents

大多数其他 Managed Agents cookbook 关注的是智能体循环本身，即让智能体针对示例数据完成有用的工作。本篇讲的是围绕这个循环的机制，也就是把 Managed Agents 应用放到真实用户面前之前所需要的部件：

1. **MCP 工具集**，用于替代自定义工具，适用于智能体需要访问 SaaS API 而不必把每次调用都经由你的应用往返一次的情况。
2. **密钥库**，用于存放每个终端用户的凭证，这样每个用户的 GitHub / Linear / Slack 词元都与其他人相互隔离，你的审计轨迹也保持干净。
3. **Webhook**，用于驱动人工介入工作，而不必全程保持一条长连接的 HTTP 连接。
4. **资源生命周期**动词（list、retrieve、update、archive、delete），用于管理你的工作区随时间积累的资源。
5. **推理地域固定**（`inference_geo`），适用于合规要求模型请求必须在特定区域执行的情况。

我们会构建一条端到端的流程，涉及前四项，最后再讲地域固定：为一个虚构的终端用户创建密钥库，给它挂上一个 GitHub MCP 凭证，运行一个在服务端使用该凭证的智能体会话，展示你在真实生产服务器上为驱动同一个会话而应注册的 Webhook 处理程序，并梳理事后清理时你会用到的管理动词。

本笔记本需要在环境中设置 `GITHUB_TOKEN`。

In [ ]:
import os

from anthropic import Anthropic
from utilities import stream_until_end_turn, wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
client = Anthropic()

GH_TOKEN = os.environ.get("GITHUB_TOKEN")
if not GH_TOKEN:
    raise SystemExit("Set GITHUB_TOKEN to run this notebook.")

## 概念：MCP 工具集与密钥库

**MCP 工具集**是继自定义工具（门控笔记本）和 `resources=` 挂载（编排笔记本）之后的第三种扩展模式。MCP 工具集把智能体指向一个实现了 [Model Context Protocol](https://modelcontextprotocol.io/) 的外部服务器。智能体在沙箱内部直接调用该服务器上的工具，无需经由你的应用往返一次：调用由 Anthropic 转发，服务器响应，智能体继续执行。绝大多数公共 SaaS API（GitHub、Slack、Linear、Stripe、Notion、Salesforce、Asana……）要么已经有 MCP 服务器，要么半天就能包上一个，它们都是合适的 MCP 候选。

经验法则：如果该服务可以用 bearer 词元通过公共互联网访问，MCP 工具集就能用。如果它只能从你自己的网络内部访问，就改用自定义工具，这正是门控笔记本所讲的内容。

**密钥库**回答的是“我该把词元放在哪里”这个问题。在创建会话时硬编码单个词元，对单一租户的部署是可行的，但一旦有终端用户就会失效。每个用户都需要自己的 GitHub 凭证，而且你必须让它们彼此隔离。密钥库是按用户划分的凭证容器，你注册一次，之后在为该用户创建的每个会话中按 ID 引用它。你不必自建密钥存储，不必在每个请求上传递词元，而且审计轨迹与密钥库绑定，所以你始终知道智能体是在为哪个终端用户工作。

## 1. 为终端用户创建密钥库

密钥库有一个在 Console 中显示的 `display_name`，以及一个 `metadata` 字典，你通常会在里面存放自己的内部用户 ID，以便把密钥库映射回你自己数据库中的一条记录。

In [ ]:
vault = client.beta.vaults.create(
    display_name="Cookbook demo user",
    metadata={"internal_user_id": "u_demo_001", "team": "engineering"},
)
print(f"vault: {vault.id}")

## 2. 挂上一个 MCP 凭证

凭证位于密钥库之下。每个凭证把一个 MCP 服务器 URL 与一个智能体调用该服务器时使用的词元配对。对于 GitHub Copilot MCP 服务器，静态 bearer 词元（你的 GitHub PAT）是最简单的形式。API 也支持带刷新机制的完整 OAuth 流程，以满足需要它的服务，两种形态都通过 `auth=` 处理。

In [ ]:
credential = client.beta.vaults.credentials.create(
    vault_id=vault.id,
    display_name="GitHub Copilot",
    auth={
        "type": "static_bearer",
        "mcp_server_url": "https://api.githubcopilot.com/mcp/",
        "token": GH_TOKEN,
    },
)
print(f"credential: {credential.id}")

## 3. 在会话中引用密钥库

在 `sessions.create` 上传递 `vault_ids=[vault.id]`，API 会在每次工具使用时查找匹配的 MCP 服务器 URL。智能体永远看不到词元本身，你也不必在请求上传递它。智能体定义照常列出 MCP 服务器即可，凭证的接线工作在创建会话时完成。

In [ ]:
agent = client.beta.agents.create(
    name="cookbook-operate",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-operate-in-production"},
    model=MODEL,
    system="You navigate GitHub repositories on behalf of the logged-in user.",
    mcp_servers=[
        {
            "type": "url",
            "name": "github",
            "url": "https://api.githubcopilot.com/mcp/",
        }
    ],
    tools=[
        {
            "type": "mcp_toolset",
            "mcp_server_name": "github",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

env = client.beta.environments.create(
    name="cookbook-operate-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    vault_ids=[vault.id],
    title="Operate demo",
)
print(f"session: {session.id}")

## 4. 以终端用户的身份运行一轮

此后智能体对 GitHub 所做的一切都经由该密钥库的凭证。在你自己的系统中做审计很直接：因为密钥库通过你在第 1 步设置的 metadata 与该用户绑定，所以你确切知道是哪个终端用户在行动。

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "List the three most recently updated repos in the anthropics org.",
                }
            ],
        }
    ],
)
print("--- vault-backed MCP call ---")
stream_until_end_turn(client, session.id)

## 5. 面向生产环境人工介入的 Webhook

门控笔记本中的流式模式在开发期间很方便，因为一切都发生在同一个进程里，但它会在人工评审期间一直占用一条 HTTP 连接，这种方式无法扩展，也无法在进程重启后存活。生产环境的模式改为在 Console 中注册一个 Webhook，在 `session.status_idled` 时触发，该信号表示智能体要么已完成，要么正在等待工具结果。

Webhook 触发时，你的服务器检查事件，把任何待处理的升级事项呈给评审人，并在人工完成时 POST 回 `user.custom_tool_result`。会话只是保持空闲直到你响应，你这边没有长连接。

如果你的会话带有 `budget`（见 [﷐1﷑](CMA_cap_session_spend.ipynb)），还应注册 `session.budget_reached`。它会在会话的清单成本达到上限时触发一次，也是监督进程所需要的信号：由它按会话逐个决定是提高上限还是让工作保持暂停。它带有会话 ID，所以处理程序与空闲事件的处理程序形态相同：查找会话、决定、执行动作。

Webhook 注册是一次性的 Console 步骤，位于 **Settings → Webhooks** 下。你会得到一个 `whsec_...` 签名密钥，它只在创建时显示一次；请把它存入你的密钥管理系统。

**下面这块是参考实现，不是笔记本单元格。**把它复制到你自己的服务器。它依赖 FastAPI，而 cookbook 并不安装 FastAPI，也不会作为本笔记本流程的一部分运行。配合门控笔记本中的智能体定义，它足以从生产服务器端到端地驱动门控工作流。

```python
import hmac
import hashlib
import json

from fastapi import FastAPI, Header, HTTPException, Request

app = FastAPI()
WEBHOOK_SECRET = "whsec_..."  # from Console, load from your secrets manager


def verify(body: bytes, sig: str) -> bool:
    expected = hmac.new(WEBHOOK_SECRET.encode(), body, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, sig)


@app.post("/webhooks/anthropic")
async def receive(req: Request, x_anthropic_signature: str = Header()):
    body = await req.body()
    if not verify(body, x_anthropic_signature):
        raise HTTPException(401)

    event = json.loads(body)
    session_id = event["resource_id"]

    if event["event_type"] == "session.status_idled":
        # Agent went idle. Either it's done OR it called escalate()
        # and is waiting on a user.custom_tool_result. Look at the
        # latest events to decide.
        events = client.beta.sessions.events.list(session_id=session_id)
        pending = [
            e for e in events.data
            if e.type == "agent.custom_tool_use" and e.name == "escalate"
            and not has_result(events, e.id)
        ]
        if pending:
            for tu in pending:
                enqueue_for_review(session_id, tu.id, tu.input)
        else:
            finalize_run(session_id)

    elif event["event_type"] == "session.budget_reached":
        # Session paused at its spend cap with its work intact. Raise
        # the budget to let it continue, or leave it paused and page
        # an operator.
        review_budget(session_id)

    return {"ok": True}


# Called from your review UI once the human decides
def submit_review(session_id: str, custom_tool_use_id: str, decision: str):
    client.beta.sessions.events.send(
        session_id=session_id,
        events=[{
            "type": "user.custom_tool_result",
            "custom_tool_use_id": custom_tool_use_id,
            "content": [{"type": "text",
                         "text": json.dumps({"human_decision": decision})}],
        }],
    )
```

响应智能体的代码与门控笔记本 A 部分的循环完全相同。唯一变化的是你的服务器如何得知有工作要做：不再是本地循环拉取事件，而是 Webhook 按你的节奏推送通知。

## 6. 固定推理地域

当合规要求模型请求必须由特定地域提供服务时，就在智能体上固定它，而不要依赖工作区默认值：

```python
agent = client.beta.agents.create(
    name="residency-pinned agent",
    model={"id": MODEL, "inference_geo": "us"},
    ...
)
```

可接受的值是 `"global"` 和 `"us"`，并受工作区的 `allowed_inference_geos` 约束。未设置时，每次请求在服务时解析为工作区的 `default_inference_geo`，因此该默认值后来的变更会波及已经在运行的会话。已设置时，该固定在保存智能体时、创建会话时以及每一轮都会校验：收紧工作区允许列表会阻止来自现已不被允许的智能体的新会话，并拒绝正在运行的会话的下一轮，而不是豁免它。正是这种严格性让该固定可作为数据驻留控制手段。`sessions.create` 上的 `agent_with_overrides` 可把单个会话移到另一个允许的地域，而不必编辑共享的智能体。完整讲解，包括会清除该固定的更新语义，见 [﷐6﷑](CMA_pin_inference_geo.ipynb)。

## 7. 资源生命周期：list、retrieve、update、archive

API 中的每种资源——智能体、环境、会话、密钥库、凭证——都暴露同样的五动词模式：`list`、`retrieve`、`update`、`archive`，以及（对某些资源）`delete`。我们先在智能体上演示完整的一组，然后把每种其他资源可用的动词列为速查表。

**archive 与 delete 的区别：**`archive` 为审计和检索保留记录，但会拆除任何运行中的容器，并停止该资源计入你的工作区配额。`delete` 则彻底移除记录。对大多数工作流而言 `archive` 是正确选择；只有在你确实需要记录消失时（例如测试清理）才使用 `delete`。

In [ ]:
listed = client.beta.agents.list(limit=5)
print(f"workspace has at least {len(listed.data)} agents")

retrieved = client.beta.agents.retrieve(agent.id)
print(f"retrieved agent: name={retrieved.name} version={retrieved.version}")

# Updating an agent produces a new version. Pass the current
# version to confirm you're updating from a known state, a
# concurrent update from another process will reject yours rather
# than silently overwriting.
updated = client.beta.agents.update(
    agent.id,
    version=agent.version,
    system="You navigate GitHub repositories. Be terse.",
)
print(f"updated to version {updated.version}")

# Every historical version is queryable.
versions = client.beta.agents.versions.list(agent_id=agent.id)
print(f"agent has {len(versions.data)} versions")

## 8. 清理

凭证和密钥库各有自己的 archive 端点。归档密钥库不会自动归档其中的凭证，所以想要彻底清理，应先处理凭证。

In [ ]:
wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
client.beta.vaults.credentials.archive(credential.id, vault_id=vault.id)
client.beta.vaults.archive(vault.id)
print("archived")

## 其他 cookbook

本笔记本是生产形态的收尾篇。它所环绕的那些工作流笔记本，如果还没跑过，值得先跑一遍：

- [﷐0﷑](CMA_iterate_fix_failing_tests.ipynb) —— 入口笔记本。通过在失败的测试套件上做观察—修复循环，介绍智能体、环境、会话、文件挂载和流式事件循环。
- [﷐0﷑](CMA_orchestrate_issue_to_pr.ipynb) —— 多轮智能体，通过模拟的 gh CLI 把一个 issue 一路推进到合并的 PR，并在中途从 CI 失败和评审意见中恢复。
- [﷐0﷑](CMA_explore_unfamiliar_codebase.ipynb) —— 扎根模式，并埋有一个过时文档的陷阱。还演示了用 `sessions.resources.add` 向运行中的会话推送更多上下文。
- [﷐0﷑](CMA_gate_human_in_the_loop.ipynb) —— 用于人工介入工作流的自定义工具 `decide()` 和 `escalate()` 往返，与上面的 Webhook 参考代码块配套使用。